In [143]:
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import subprocess
from multiprocessing import Pool
import math 
import string

In [14]:
accessions=['at6923', 'at6929', 'at7143', 'at8285', 'at9104', 'at9336', 'at9503', 'at9578', 'at9744', 'at9762', 'at9806', 'at9830', 'at9847', 'at9852', 'at9879', 'at9883', 'at9900']

In [162]:
def read(file_name):
    try:
        fasta = open(file_name, 'r')
        fasta_pairs = {}
        header = ''
        # check every line for a new header
        # if so, set as key and set non header lines as value
        for line in fasta:
            if line[0] == '>':
                header = line.strip('\n')
                fasta_pairs[header] = ''
            else:
                fasta_pairs[header] += line.strip('\n')
        # return dict of fasta entries
        return fasta_pairs
    except IOError:
        print("File \"{}\" could not be opened.".format(file_name))

In [163]:
def get_nucleotides(chrom,strand, start, end, distance):
    nn = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    if strand == "+":
        donor_context=chrom_seq[start+1-distance:start+1]
        donor_context_start=start+2-distance
        donor_context_end=start+1
        acceptor_context=chrom_seq[end-2:end+distance-2]
        acceptor_context_start=end-1
        acceptor_context_end=end+distance-2
    else:
        donor_context=rev_compl(chrom_seq[end-2:end+distance-2])
        donor_context="".join(nn[n] for n in reversed(dono))
        donor_context_start=end-1
        donor_context_end=end+distance-2
        acceptor_context=rev_compl(chrom_seq[start+1-distance:start+1])
        acceptor_context_start=start+2-distance
        acceptor_context_end=start+1
    return donor_context, donor_context_start, donor_context_end, acceptor_context, acceptor_context_start, acceptor_context_end

In [204]:
def junction_window_size(acc):
    distance=4
    genome=read(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/input/refs/{acc}.scaffolds-v2.3.fasta")
    junctions=pd.read_csv(f"output/04/{acc}/{acc}-hodgepodgemerged_multi-acceptors-hodgepodge_geneIDs.bed", sep="\t")
    acc_df=pd.DataFrame()
    for chromosome in junctions.groupby(by="chrom"):
        chrom = genome[">"+chromosome[0]]
        for junction in chromosome[1].groupby(by="junction_id"):
            #get function info
            j_df=junction[1]
            strand=j_df["strand"].values[0]
            start=j_df["genomic_start_coord"].values[0]
            end=j_df["genomic_end_coord"].values[0]
            #call function
            donor_context, donor_context_start, donor_context_end, acceptor_context, acceptor_context_start, acceptor_context_end = get_nucleotides(chrom, strand, start, end, distance)
            j_df["donor_context"]=donor_context
            j_df["donor_context_start"]=donor_context_start
            j_df["donor_context_end"]=donor_context_end
            j_df["acceptor_context"]=acceptor_context
            j_df["acceptor_context_start"]=acceptor_context_start
            j_df["acceptor_context_end"]=acceptor_context_end
            acc_df=pd.concat([acc_df,j_df])
    acc_df.to_csv(f"output/10/{acc}/{acc}-hodgepodgemerged_junction_window_size-{distance}.bed", sep="\t", index=False)

In [205]:
if __name__ == '__main__':
    with Pool(17) as p:
        print(p.map(junction_window_size, accessions))

[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
